In [1]:
from aocd import get_data
data_real = get_data(day=7, year=2024)
data_real[:40]

'9151: 132 1 8 714 972 5 21 1\n740: 136 4 '

# part 1

When you go to cross the bridge, you notice a group of engineers trying to repair it. (Apparently, it breaks pretty frequently.) You won't be able to cross until it's fixed.

You ask how long it'll take; the engineers tell you that it only needs final calibrations, but some young elephants were playing nearby and stole all the operators from their calibration equations! They could finish the calibrations if only someone could determine which test values could possibly be produced by placing any combination of operators into their calibration equations (your puzzle input).

For example:

```
190: 10 19
3267: 81 40 27
83: 17 5
156: 15 6
7290: 6 8 6 15
161011: 16 10 13
192: 17 8 14
21037: 9 7 18 13
292: 11 6 16 20
```

Each line represents a single equation. The test value appears before the colon on each line; it is your job to determine whether the remaining numbers can be combined with operators to produce the test value.

Operators are always evaluated left-to-right, not according to precedence rules. Furthermore, numbers in the equations cannot be rearranged. Glancing into the jungle, you can see elephants holding two different types of operators: add (+) and multiply (*).

In [2]:
data_test = """190: 10 19
3267: 81 40 27
83: 17 5
156: 15 6
7290: 6 8 6 15
161011: 16 10 13
192: 17 8 14
21037: 9 7 18 13
292: 11 6 16 20"""

In [7]:
[row.split(" ") for row in data_test.splitlines()]

[['190:', '10', '19'],
 ['3267:', '81', '40', '27'],
 ['83:', '17', '5'],
 ['156:', '15', '6'],
 ['7290:', '6', '8', '6', '15'],
 ['161011:', '16', '10', '13'],
 ['192:', '17', '8', '14'],
 ['21037:', '9', '7', '18', '13'],
 ['292:', '11', '6', '16', '20']]

In [10]:
equations = [[int(row.split()[0].strip(':')), [int(x) for x in row.split()[1:]]] for row in data_test.splitlines()]
equations[:3]

[[190, [10, 19]], [3267, [81, 40, 27]], [83, [17, 5]]]

In [75]:
def doOp(name, args):
    if name == "+": return sum(args)
    elif name == "*":
        result = 1
        for arg in args: result *= arg
        return result
doOp("+", [1,2,3]), doOp("*", [1,2,3])

(6, 6)

In [80]:
from itertools import product

equations = [[int(row.split()[0].strip(':')), [int(x) for x in row.split()[1:]]] for row in data_test.splitlines()]

sumout=0
for result, inputs in equations:
    pairs = [(inputs[i], inputs[i+1]) for i in range(len(inputs)-1)]
    ops=list(product(["+","*"], repeat=len(inputs)-1))
    results = []
    for o in ops:
        result_ = inputs[0]
        for argnum,arg in enumerate(inputs[1:]):
            result_ = doOp(o[argnum], [result_, arg])
        results.append(result_)
    if result in results: sumout += result
    print(result,inputs, result in results)
sumout

190 [10, 19] True
3267 [81, 40, 27] True
83 [17, 5] False
156 [15, 6] False
7290 [6, 8, 6, 15] False
161011 [16, 10, 13] False
192 [17, 8, 14] False
21037 [9, 7, 18, 13] False
292 [11, 6, 16, 20] True


3749

works, let's try the real data

In [82]:
from itertools import product

equations = [[int(row.split()[0].strip(':')), [int(x) for x in row.split()[1:]]] for row in data_real.splitlines()]

sumout=0
for result, inputs in equations:
    pairs = [(inputs[i], inputs[i+1]) for i in range(len(inputs)-1)]
    ops=list(product(["+","*"], repeat=len(inputs)-1))
    results = []
    for o in ops:
        result_ = inputs[0]
        for argnum,arg in enumerate(inputs[1:]):
            result_ = doOp(o[argnum], [result_, arg])
        results.append(result_)
    if result in results: sumout += result
    # print(result,inputs, result in results)
sumout

10741443549536

# part 2 

The engineers seem concerned; the total calibration result you gave them is nowhere close to being within safety tolerances. Just then, you spot your mistake: some well-hidden elephants are holding a third type of operator.

The **concatenation operator (||)** combines the digits from its left and right inputs into a single number. For example, 12 || 345 would become 12345. All operators are still evaluated left-to-right.

Now, apart from the three equations that could be made true using only addition and multiplication, the above example has three more equations that can be made true by inserting operators:

156: 15 6 can be made true through a single concatenation: 15 || 6 = 156.
7290: 6 8 6 15 can be made true using 6 * 8 || 6 * 15.
192: 17 8 14 can be made true using 17 || 8 + 14.
Adding up all six test values (the three that could be made before using only + and * plus the new three that can now be made by also using ||) produces the new total calibration result of 11387.

Using your new knowledge of elephant hiding spots, determine which equations could possibly be true. What is their total calibration result?

In [85]:
def doOp(name, args):
    if name == "+": return sum(args)
    elif name == "*":
        result = 1
        for arg in args: result *= arg
        return result
    elif name == "||":
        result = ""
        for arg in args: result += str(arg)
        return int(result)
doOp("+", [1,2,3]), doOp("*", [1,2,3]), doOp("||", [1,235,4])

(6, 6, 12354)

In [86]:
from itertools import product

equations = [[int(row.split()[0].strip(':')), [int(x) for x in row.split()[1:]]] for row in data_test.splitlines()]

good_results=[]
for result, inputs in equations:
    pairs = [(inputs[i], inputs[i+1]) for i in range(len(inputs)-1)]
    ops=list(product(["+","*","||"], repeat=len(inputs)-1))
    results = []
    for o in ops:
        result_ = inputs[0]
        for argnum,arg in enumerate(inputs[1:]):
            result_ = doOp(o[argnum], [result_, arg])
        results.append(result_)
    if result in results: good_results.append(result)
    print(result,inputs, result in results)
len(good_results), sum(good_results)

190 [10, 19] True
3267 [81, 40, 27] True
83 [17, 5] False
156 [15, 6] True
7290 [6, 8, 6, 15] True
161011 [16, 10, 13] False
192 [17, 8, 14] True
21037 [9, 7, 18, 13] False
292 [11, 6, 16, 20] True


(6, 11387)

In [88]:
from itertools import product
from tqdm import tqdm

equations = [[int(row.split()[0].strip(':')), [int(x) for x in row.split()[1:]]] for row in data_real.splitlines()]

good_results=[]
for result, inputs in tqdm(equations):
    pairs = [(inputs[i], inputs[i+1]) for i in range(len(inputs)-1)]
    ops=list(product(["+","*","||"], repeat=len(inputs)-1))
    results = []
    for o in ops:
        result_ = inputs[0]
        for argnum,arg in enumerate(inputs[1:]):
            result_ = doOp(o[argnum], [result_, arg])
        results.append(result_)
    if result in results: good_results.append(result)
    # print(result,inputs, result in results)
len(good_results), sum(good_results)

100%|███████████████████████████████████████████████████| 850/850 [00:35<00:00, 23.84it/s]


(618, 500335179214836)

In [90]:
%%time

from itertools import product
from tqdm import tqdm

equations = [[int(row.split()[0].strip(':')), [int(x) for x in row.split()[1:]]] for row in data_real.splitlines()]

good_results=[]
for result, inputs in tqdm(equations):
    pairs = [(inputs[i], inputs[i+1]) for i in range(len(inputs)-1)]
    ops=list(product(["+","*","||"], repeat=len(inputs)-1))
    results = []
    for o in ops:
        skipThis = False
        result_ = inputs[0]
        for argnum,arg in enumerate(inputs[1:]):
            result_ = doOp(o[argnum], [result_, arg])
            if result_>result: skipThis=True; break
        if not skipThis: results.append(result_)
    if result in results: good_results.append(result)
    # print(result,inputs, result in results)
len(good_results), sum(good_results)

100%|███████████████████████████████████████████████████| 850/850 [00:33<00:00, 25.49it/s]

CPU times: user 33.3 s, sys: 52.4 ms, total: 33.3 s
Wall time: 33.4 s


(618, 500335179214836)

In [92]:
def doOp(name, args):
    a1, a2 = args
    if name == "+": return a1+a2
    elif name == "*": return a1*a2
    elif name == "||": return int(str(a1)+str(a2))
doOp("+", [1,2]), doOp("*", [1,2]), doOp("||", [1,235])

(3, 2, 1235)

In [93]:
%%time

from itertools import product
from tqdm import tqdm

equations = [[int(row.split()[0].strip(':')), [int(x) for x in row.split()[1:]]] for row in data_real.splitlines()]

good_results=[]
for result, inputs in tqdm(equations):
    pairs = [(inputs[i], inputs[i+1]) for i in range(len(inputs)-1)]
    ops=list(product(["+","*","||"], repeat=len(inputs)-1))
    results = []
    for o in ops:
        skipThis = False
        result_ = inputs[0]
        for argnum,arg in enumerate(inputs[1:]):
            result_ = doOp(o[argnum], [result_, arg])
            if result_>result: skipThis=True; break
        if not skipThis: results.append(result_)
    if result in results: good_results.append(result)
    # print(result,inputs, result in results)
len(good_results), sum(good_results)

100%|███████████████████████████████████████████████████| 850/850 [00:29<00:00, 28.69it/s]

CPU times: user 29.6 s, sys: 47.9 ms, total: 29.6 s
Wall time: 29.6 s


(618, 500335179214836)

shit, there's still an old remnant in there from the initial try

In [ ]:
from itertools import product
from tqdm import tqdm

In [94]:
%%time

equations = [[int(row.split()[0].strip(':')), [int(x) for x in row.split()[1:]]] for row in data_real.splitlines()]

good_results=[]
for result, inputs in equations:
    ops=list(product(["+","*","||"], repeat=len(inputs)-1))
    results = []
    for o in ops:
        skipThis = False
        result_ = inputs[0]
        for argnum,arg in enumerate(inputs[1:]):
            result_ = doOp(o[argnum], [result_, arg])
            if result_>result: skipThis=True; break
        if not skipThis: results.append(result_)
    if result in results: good_results.append(result)
    # print(result,inputs, result in results)
len(good_results), sum(good_results)

CPU times: user 29.6 s, sys: 35.2 ms, total: 29.7 s
Wall time: 29.8 s


(618, 500335179214836)

In [98]:
def doOp(name, a1, a2):
    # a1, a2 = args
    if name == "+": return a1+a2
    elif name == "*": return a1*a2
    elif name == "||": return int(str(a1)+str(a2))
doOp("+", 1, 2), doOp("*", 1, 2), doOp("||", 1, 235)

(3, 2, 1235)

In [99]:
%%time

equations = [[int(row.split()[0].strip(':')), [int(x) for x in row.split()[1:]]] for row in data_real.splitlines()]

good_results=[]
for result, inputs in equations:
    ops=list(product(["+","*","||"], repeat=len(inputs)-1))
    results = []
    for o in ops:
        skipThis = False
        result_ = inputs[0]
        for argnum,arg in enumerate(inputs[1:]):
            result_ = doOp(o[argnum], result_, arg)
            if result_>result: skipThis=True; break
        if not skipThis: results.append(result_)
    if result in results: good_results.append(result)
    # print(result,inputs, result in results)
len(good_results), sum(good_results)

CPU times: user 26.1 s, sys: 26.5 ms, total: 26.2 s
Wall time: 26.2 s


(618, 500335179214836)

In [106]:
%%time

equations = [[int(row.split()[0].strip(':')), [int(x) for x in row.split()[1:]]] for row in data_real.splitlines()]

good_results=0

for result, inputs in equations:
    ops=list(product(["+","*","||"], repeat=len(inputs)-1))
    for o in ops:
        result_ = inputs[0]
        for argnum,arg in enumerate(inputs[1:]): result_ = doOp(o[argnum], result_, arg)
        if result_ == result: good_results+1; break

good_results

100%|███████████████████████████████████████████████████| 850/850 [00:15<00:00, 53.22it/s]

CPU times: user 15.9 s, sys: 31.8 ms, total: 15.9 s
Wall time: 16 s


0